# XGBoost From First Principles: Fill-in-the-Gaps Notebook

This notebook is a guided build of the core ideas behind XGBoost. It is intentionally incomplete: code cells contain `TODO` blocks for you to fill in.

The goal is not to reproduce every engineering feature of the real `xgboost` library. The goal is to understand the mathematical engine:

1. Boosting as additive modeling.
2. Loss functions, gradients, and Hessians.
3. Why XGBoost fits trees to gradient/Hessian statistics instead of raw labels.
4. How leaf weights are chosen analytically.
5. How split gain decides whether a tree split is worth making.
6. How shrinkage, regularization, and tree depth affect learning.

Suggested prerequisites to read up on as you go: derivatives, Taylor approximations, decision trees, regularization, gradient descent, and logistic regression.

## How to Use This Notebook

Work top to bottom. When you see a function with a `TODO`, pause and implement it before running the following check cell.

A good rhythm:

- Read the intuition cell.
- Predict what the formula should do in plain English.
- Fill the code skeleton.
- Run the check cell.
- Only then move on.

If a check fails, inspect shapes first. Most bugs in first-principles ML code are either sign errors or array-shape mismatches.

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)

## 1. A Tiny Regression Problem

We will begin with regression because the squared-error loss has simple gradients and Hessians.

The data is one-dimensional on purpose. That lets you plot the model and see boosting happen. Real XGBoost handles many features, missing values, categorical encodings, sparsity, and lots of systems tricks. We are building the conceptual core first.

In [ ]:
n = 180
X = np.sort(rng.uniform(-3, 3, size=(n, 1)), axis=0)
y_true = np.sin(1.4 * X[:, 0]) + 0.35 * X[:, 0]
y = y_true + rng.normal(0, 0.18, size=n)

plt.figure(figsize=(8, 4))
plt.scatter(X[:, 0], y, s=18, alpha=0.75, label="observed")
plt.plot(X[:, 0], y_true, color="black", linewidth=2, label="noise-free pattern")
plt.legend()
plt.title("Toy regression data")
plt.show()

## 2. Boosting as Additive Modeling

A boosted model is a sum of small models:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta f_t(x_i)$$

where:

- $f_t$ is the new tree added at round $t$.
- $\eta$ is the learning rate, also called shrinkage.
- Earlier trees stay fixed; each new tree tries to improve the current predictions.

For squared error, the best constant first prediction is the mean of the labels. Your first task is to compute that baseline.

In [ ]:
def initial_prediction_squared_error(y):
    """Return the best constant prediction for squared-error regression."""
    # TODO: replace None with the value that minimizes mean((y - c)^2).
    return None


base_pred = initial_prediction_squared_error(y)
y_hat0 = np.full_like(y, fill_value=base_pred, dtype=float)
base_pred

In [ ]:
# Check: the best constant squared-error prediction is the sample mean.
assert np.isclose(base_pred, np.mean(y))
print("Baseline prediction looks right:", base_pred)

## 2.5 What Is the Next Tree Actually Trying to Do?

Before we talk about gradients and Hessians formally, build this picture first.

At a given boosting round, the model already has a prediction for every training sample. The old trees are frozen. XGBoost is asking a narrower question:

> What should the next tree output for each sample so the model prediction moves in a better direction?

If a sample has true value $y_i = 10$ and the current model predicts $\hat{y}_i = 7$, then the model is too low. A helpful next output would push the prediction upward.

For squared-error regression, the natural correction is:

$$y_i - \hat{y}_i$$

So in this example the correction is $10 - 7 = 3$.

Keep this mental model: each training sample is asking for a correction. The next tree tries to approximate those correction requests.

In [ ]:
# Tiny sign exercise: before using calculus, reason about correction direction.
example_y = np.array([10.0, 4.0, -2.0])
example_y_hat = np.array([7.0, 6.0, -5.0])

def desired_squared_error_correction(y, y_hat):
    """For squared error, return the direct residual-like correction y - y_hat."""
    # TODO: compute the direction each prediction should move.
    return None


example_correction = desired_squared_error_correction(example_y, example_y_hat)
example_correction

In [ ]:
assert np.allclose(example_correction, np.array([3.0, -2.0, 3.0]))
print("Positive means push prediction up; negative means push it down:", example_correction)

## 2.6 Why a Tree Cannot Give Every Sample Its Own Perfect Correction

A tree does not usually assign an independent output to every training row. It partitions feature space into leaves, and every sample in the same leaf gets the same output value.

That means the tree has to compromise.

Suppose a leaf contains samples whose desired corrections are:

$$[3, 2, 4]$$

A single shared leaf value around $3$ makes sense. But if a leaf contains:

$$[3, -8, 4]$$

then one shared value is awkward. A split may help, because one child leaf can handle positive corrections while another handles negative corrections.

This is the intuition behind split search: find feature thresholds that group together samples wanting similar prediction movements.

In [ ]:
def simple_leaf_correction(corrections):
    """A pre-calculus version of a leaf output: average the requested corrections."""
    # TODO: return the average correction for this leaf.
    return None


friendly_leaf = np.array([3.0, 2.0, 4.0])
awkward_leaf = np.array([3.0, -8.0, 4.0])

print("Friendly shared correction:", simple_leaf_correction(friendly_leaf))
print("Awkward shared correction:", simple_leaf_correction(awkward_leaf))

In [ ]:
assert np.isclose(simple_leaf_correction(friendly_leaf), 3.0)
assert np.isclose(simple_leaf_correction(awkward_leaf), -1.0 / 3.0)
print("Averaging corrections is reasonable only when the leaf groups compatible samples.")

## 2.7 Where Gradients and Hessians Enter the Story

The correction story above is exact-looking only because squared error is especially simple.

For more general objectives, XGBoost does not simply ask for `y - y_hat`. Instead, for each training sample it computes two local quantities at the current prediction:

- The gradient says which direction changes the objective.
- The Hessian says how curved or sensitive the objective is around the current prediction.

This is different from neural-network training. We are not taking gradients with respect to tree weights in a big differentiable tree. The per-sample gradients and Hessians are taken with respect to the current model prediction.

Then the next tree is built using those per-sample numbers. Informally:

- Samples with similar correction needs should end up together.
- Each leaf gets one shared output.
- Gradients tell the leaf which way to move.
- Hessians tell the leaf how strongly to trust or weight those moves.

Now we are ready to make that intuition precise with a loss function.

## 3. Loss, Gradients, and Hessians

XGBoost works with an objective. For each point, there is a loss $\ell(y_i, \hat{y}_i)$.

At each boosting round, XGBoost asks: if I add a small tree output to the current prediction, how will the loss change?

It answers this using a second-order Taylor approximation:

$$\ell(y_i, \hat{y}_i + f_i) \approx \ell(y_i, \hat{y}_i) + g_i f_i + \frac{1}{2} h_i f_i^2$$

where:

- $g_i = \partial_{\hat{y}} \ell(y_i, \hat{y}_i)$ is the gradient.
- $h_i = \partial^2_{\hat{y}} \ell(y_i, \hat{y}_i)$ is the Hessian.

For squared error we will use:

$$\ell(y, \hat{y}) = \frac{1}{2}(\hat{y} - y)^2$$

The factor $1/2$ makes the derivative cleaner.

In [ ]:
def squared_error_loss(y, y_hat):
    """Return the average 1/2 squared-error loss."""
    # TODO: implement 0.5 * avg((y_hat - y)^2)
    return None


def squared_error_grad_hess(y, y_hat):
    """Return gradient and Hessian of 1/2 squared error with respect to y_hat."""
    # TODO: derive these by differentiating 0.5 * (y_hat - y)^2.
    grad = None
    hess = None
    return grad, hess

In [ ]:
g0, h0 = squared_error_grad_hess(y, y_hat0)

assert np.isclose(squared_error_loss(np.array([2.0]), np.array([5.0])), 4.5)
assert np.allclose(g0, y_hat0 - y)
assert np.allclose(h0, np.ones_like(y))
print("Squared-error loss, gradient, and Hessian look right.")

## 4. The Leaf Weight Formula
At boosting round $t$, assume the current model prediction for point $i$ is

$$
\hat{y}_i = F_{t-1}(x_i)
$$

Now we add a new tree $f_t(x)$, so the new prediction becomes

$$
F_t(x_i) = F_{t-1}(x_i) + f_t(x_i)
$$

or equivalently,

$$
\hat{y}_i^{new} = \hat{y}_i + f_t(x_i)
$$

Therefore, the loss after adding the new tree is

$$
\ell(y_i, \hat{y}_i + f_t(x_i))
$$

This expression may be hard to optimize directly, so XGBoost approximates it using a second-order Taylor expansion around the current prediction $\hat{y}_i$.

Define a temporary function

$$
q_i(z) = \ell(y_i, \hat{y}_i + z)
$$

where $z$ represents the amount by which the new tree changes the current prediction.

Before adding the new tree, $z=0$. After adding the new tree,

$$
z = f_t(x_i)
$$

Now apply the second-order Taylor expansion of $q_i(z)$ around $z=0$:

$$
q_i(z) \approx q_i(0) + q_i'(0)z + \frac{1}{2}q_i''(0)z^2
$$

Substituting back:

$$
q_i(0) = \ell(y_i, \hat{y}_i)
$$

The first derivative is

$$
q_i'(0)
=
\left.\frac{\partial}{\partial z}\ell(y_i, \hat{y}_i + z)\right|_{z=0}
=
\frac{\partial}{\partial \hat{y}_i}\ell(y_i, \hat{y}_i)
=
g_i
$$

The second derivative is

$$
q_i''(0)
=
\left.\frac{\partial^2}{\partial z^2}\ell(y_i, \hat{y}_i + z)\right|_{z=0}
=
\frac{\partial^2}{\partial \hat{y}_i^2}\ell(y_i, \hat{y}_i)
=
h_i
$$

Therefore,

$$
\ell(y_i, \hat{y}_i + f_t(x_i))
\approx
\ell(y_i, \hat{y}_i)
+
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
$$

Now suppose we focus on one specific leaf of the new tree.

For every point $i$ that lands in this leaf, the tree outputs the same constant value $w$:

$$
f_t(x_i) = w
$$

So for every point in this leaf,

$$
\ell(y_i, \hat{y}_i + f_t(x_i))
\approx
\ell(y_i, \hat{y}_i)
+
g_i w
+
\frac{1}{2}h_i w^2
$$

When choosing the new tree, the term

$$
\ell(y_i, \hat{y}_i)
$$

does not depend on the new tree. It is already fixed by the previous model. Therefore, we can ignore it for optimization.

So the part of the approximate objective contributed by the samples in this leaf is

$$
\sum_{i \in leaf}
\left(
g_i w + \frac{1}{2}h_i w^2
\right)
$$

XGBoost also adds an $L_2$ regularization penalty on the leaf value $w$:

$$
\frac{1}{2}\lambda w^2
$$
Imagine a tree leaf contains a set of training points. The tree predicts one constant value $w$ for every point in that leaf.

Using the Taylor approximation, the part of the objective affected by that leaf is:

$$\sum_{i \in leaf}(g_i w + \frac{1}{2}h_i w^2) + \frac{1}{2}\lambda w^2$$

Group the gradients and Hessians:

$$G = \sum_i g_i, \quad H = \sum_i h_i$$

Then the leaf objective is:

$$G w + \frac{1}{2}(H + \lambda)w^2$$

This is just a quadratic. The minimizing value is:

$$w^* = -\frac{G}{H + \lambda}$$

This is one of the most important ideas in XGBoost: once a tree structure is chosen, the best leaf values are available in closed form.

In [ ]:
def leaf_weight(grad, hess, reg_lambda=1.0):
    """Return the optimal XGBoost leaf value for the points in one leaf."""
    G = None
    H = None
    # TODO: implement -G / (H + reg_lambda)
    return None

For one leaf, the second-order approximate objective is

$$
J(w) = G w + \frac{1}{2}(H+\lambda)w^2
$$

where

$$
G = \sum_{i \in leaf} g_i,
\qquad
H = \sum_{i \in leaf} h_i
$$

To find the best leaf value, differentiate with respect to $w$:

$$
\frac{dJ}{dw}
=
G + (H+\lambda)w
$$

Set the derivative to zero:

$$
G + (H+\lambda)w = 0
$$

so

$$
w^* = -\frac{G}{H+\lambda}
$$

Now plug $w^*$ back into the leaf objective:

$$
J(w^*)
=
G\left(-\frac{G}{H+\lambda}\right)
+
\frac{1}{2}(H+\lambda)
\left(-\frac{G}{H+\lambda}\right)^2
$$

The first term is

$$
-\frac{G^2}{H+\lambda}
$$

and the second term is

$$
\frac{1}{2}\frac{G^2}{H+\lambda}
$$

Therefore,

$$
J(w^*)
=
-\frac{1}{2}\frac{G^2}{H+\lambda}
$$

This is the best achievable approximate objective value for that leaf.

Since lower objective is better, the improvement from creating this leaf is the positive quantity

$$
\frac{1}{2}\frac{G^2}{H+\lambda}
$$

In the code, `leaf_score` returns the same score before applying the conventional $\frac{1}{2}$ factor:

$$
\text{leaf\_score}
=
\frac{G^2}{H+\lambda}
$$

In [ ]:
def leaf_score(grad, hess, reg_lambda=1.0):
    """Return the objective improvement term associated with one leaf.

    Bigger is better. This is the positive quantity G^2 / (H + lambda),
    before the conventional 1/2 factor is applied in split_gain.
    """
    G = None
    H = None
    # TODO: implement G^2 / (H + reg_lambda)
    return None

In [ ]:
toy_g = np.array([2.0, -1.0, 3.0])
toy_h = np.array([1.0, 1.0, 1.0])

assert np.isclose(leaf_weight(toy_g, toy_h, reg_lambda=1.0), -1.0)
assert np.isclose(leaf_score(toy_g, toy_h, reg_lambda=1.0), 4.0)
print("Leaf formulas look right.")

## 5. Split Gain

A decision tree split is useful only if replacing one parent leaf with two child leaves improves the regularized objective.

Suppose we are considering splitting one parent leaf into a left child leaf and a right child leaf.

For the parent leaf, define

$$
G = \sum_{i \in parent} g_i,
\qquad
H = \sum_{i \in parent} h_i
$$

For the left child leaf, define

$$
G_L = \sum_{i \in left} g_i,
\qquad
H_L = \sum_{i \in left} h_i
$$

For the right child leaf, define

$$
G_R = \sum_{i \in right} g_i,
\qquad
H_R = \sum_{i \in right} h_i
$$

Because the parent is just the union of the left and right children,

$$
G = G_L + G_R
$$

and

$$
H = H_L + H_R
$$

For any leaf, the best achievable reduction in the approximate objective is

$$
\frac{1}{2}\frac{G^2}{H+\lambda}
$$

This comes from first finding the optimal leaf value

$$
w^* = -\frac{G}{H+\lambda}
$$

and then plugging it back into the leaf objective.

Before the split, all samples are grouped together in one parent leaf, so the parent score is

$$
\frac{1}{2}\frac{G^2}{H+\lambda}
$$

After the split, the samples are separated into two leaves. The left leaf gets its own optimal value, and the right leaf gets its own optimal value. Therefore, the new score is

$$
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
\right]
$$

The gain from the split is the improvement after splitting minus the score before splitting:

$$
Gain
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right]
$$

XGBoost also subtracts a penalty for adding a new split. This penalty is denoted by $\gamma$:

$$
Gain
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right]
-
\gamma
$$

A split is useful only if this gain is positive.

Intuitively, before the split, the parent leaf must give one shared correction to all samples inside it. After the split, the left and right child leaves can each give their own correction. The split is good when these two more specialized corrections reduce the loss more than the original shared correction.

The term

$$
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
$$

measures how good the two child leaves are together.

The term

$$
\frac{G^2}{H+\lambda}
$$

measures how good the original parent leaf was before splitting.

So the split gain asks:

$$
\text{How much better are the two children than the one parent?}
$$

The regularization parameters control how eager the model is to split:

- $\lambda$ shrinks leaf values by making the denominator larger.
- $\gamma$ directly penalizes creating an additional split.

So a split is accepted only when the improvement from more specialized leaf corrections is large enough to overcome the regularization penalty.

In [ ]:
def split_gain(left_grad, left_hess, right_grad, right_hess, reg_lambda=1.0, gamma=0.0):
    """Return the XGBoost split gain for a proposed split."""
    # TODO: compute 0.5 * (left_score + right_score - parent_score) - gamma
    parent_grad = None
    parent_hess = None
    gain = None
    return gain

In [ ]:
lg = np.array([-2.0, -2.0])
lh = np.array([1.0, 1.0])
rg = np.array([2.0, 2.0])
rh = np.array([1.0, 1.0])

assert split_gain(lg, lh, rg, rh, reg_lambda=0.0, gamma=0.0) > 0
assert split_gain(lg, lh, rg, rh, reg_lambda=0.0, gamma=100.0) < 0
print("Split gain responds correctly to useful splits and gamma penalties.")

## 6. Searching for the Best Split

For one numeric feature, possible split thresholds lie between sorted adjacent feature values.

For each threshold:

- Points with `x <= threshold` go left.
- Points with `x > threshold` go right.
- We compute the gain from gradient/Hessian sums.

Real XGBoost has efficient approximate split finders and weighted quantile sketches. Here we do the clear, slow version because it teaches the idea.

In [ ]:
@dataclass
class Split:
    feature: int
    threshold: float
    gain: float


def find_best_split(X, grad, hess, reg_lambda=1.0, gamma=0.0, min_child_weight=1.0):
    """Find the best split across all numeric features.

    min_child_weight is a lower bound on the sum of Hessians in each child.
    For squared error, Hessians are ones, so it acts like a minimum leaf size.
    """
    n_samples, n_features = X.shape
    best = None

    for feature in range(n_features):
        order = np.argsort(X[:, feature])
        x_sorted = X[order, feature]
        g_sorted = grad[order]
        h_sorted = hess[order]

        for j in range(1, n_samples):
            # Do not split between identical feature values.
            if x_sorted[j] == x_sorted[j - 1]:
                continue

            threshold = 0.5 * (x_sorted[j - 1] + x_sorted[j])

            # TODO: build left/right gradient and Hessian arrays for this threshold.
            left_grad = None
            left_hess = None
            right_grad = None
            right_hess = None

            # TODO: skip splits where either child has sum(hess) < min_child_weight.
            if False:
                continue

            gain = split_gain(left_grad, left_hess, right_grad, right_hess, reg_lambda, gamma)

            if best is None or gain > best.gain:
                best = Split(feature=feature, threshold=float(threshold), gain=float(gain))

    return best

In [ ]:
# After you implement find_best_split, this should find a useful first split.
best0 = find_best_split(X, g0, h0, reg_lambda=1.0, gamma=0.0, min_child_weight=5.0)

assert best0 is not None
assert best0.gain > 0
print(best0)

## 7. From One Split to a Tree

A tree is a recursive structure:

- A leaf stores a prediction value.
- An internal node stores a feature, threshold, left child, and right child.

At each node we search for the best split. We stop when one of these happens:

- The tree reaches `max_depth`.
- No split has positive enough gain.
- A child would violate `min_child_weight`.

This is the part where the abstract gain formula turns into an actual tree learner.

In [ ]:
@dataclass
class Node:
    value: Optional[float] = None
    feature: Optional[int] = None
    threshold: Optional[float] = None
    left: Optional["Node"] = None
    right: Optional["Node"] = None

    @property
    def is_leaf(self):
        return self.value is not None


def fit_tree(X, grad, hess, depth=0, max_depth=2, reg_lambda=1.0, gamma=0.0, min_child_weight=1.0):
    """Fit one regression tree to gradient/Hessian statistics."""
    # TODO: compute this node's leaf value using leaf_weight.
    current_value = None

    # Stop if max depth has been reached.
    if depth >= max_depth:
        return Node(value=current_value)

    split = find_best_split(X, grad, hess, reg_lambda, gamma, min_child_weight)

    # TODO: if there is no useful split, return a leaf.
    if False:
        return Node(value=current_value)

    # TODO: partition rows by the split.
    left_mask = None
    right_mask = None

    left_child = fit_tree(
        X[left_mask], grad[left_mask], hess[left_mask],
        depth + 1, max_depth, reg_lambda, gamma, min_child_weight
    )
    right_child = fit_tree(
        X[right_mask], grad[right_mask], hess[right_mask],
        depth + 1, max_depth, reg_lambda, gamma, min_child_weight
    )

    return Node(feature=split.feature, threshold=split.threshold, left=left_child, right=right_child)


def predict_one_tree(node, x_row):
    """Predict with one fitted tree for a single row."""
    # TODO: recursively descend until you reach a leaf.

    return NotImplementedError


def predict_tree(node, X):
    """Predict with one fitted tree for many rows."""
    return NotImplementedError

In [ ]:
# This check should pass once fit_tree and prediction are implemented.
tree0 = fit_tree(X, g0, h0, max_depth=2, reg_lambda=1.0, gamma=0.0, min_child_weight=5.0)

In [ ]:
tree0_pred = predict_tree(tree0, X)

In [ ]:
assert tree0_pred.shape == y.shape
assert np.all(np.isfinite(tree0_pred))
assert squared_error_loss(y, y_hat0 + tree0_pred) < squared_error_loss(y, y_hat0)
print("One tree improves the baseline.")

## 8. Add Trees One at a Time

Now we have all the ingredients for a small XGBoost-like regressor.

Each round:

1. Compute gradients and Hessians at the current predictions.
2. Fit a tree to those statistics.
3. Add the tree's output to the model, scaled by the learning rate.

The learning rate matters because individual trees are greedy. Shrinkage makes each tree take a smaller step, which often improves generalization.

In [ ]:
@dataclass
class TinyXGBRegressor:
    base_score: float
    trees: list
    learning_rate: float
    loss_history: list


def fit_tiny_xgb_regressor(
    X,
    y,
    n_estimators=20,
    learning_rate=0.1,
    max_depth=2,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=1.0,
):
    """Fit a tiny XGBoost-style regressor using squared-error loss."""
    # TODO: initialize base_score and y_hat.
    base_score = leaf_score(y-y.mean(),np.ones_like(y))
    y_hat = y.mean()
    trees = []
    for round_idx in range(n_estimators):
        # TODO: compute gradient and Hessian for current predictions.
        grad, hess = y_hat-y, np.ones_like(y)

        tree = fit_tree(X, grad, hess, 0, max_depth, reg_lambda, gamma, min_child_weight)
        update = predict_tree(tree, X)

        # TODO: update predictions using learning_rate * update.
        y_hat = None
        trees.append(tree)

    return TinyXGBRegressor(base_score=base_score, trees=trees, learning_rate=learning_rate,loss_history=None)


def predict_tiny_xgb(model, X):
    """Predict with the tiny boosted tree ensemble."""
    # TODO: start with base_score and add each shrunken tree prediction.
    y_hat = model.base_score
    for tree in model.trees:
        y_hat += model.learning_rate* predict_tree(tree,X)
    return y_hat

In [ ]:
model = fit_tiny_xgb_regressor(
    X,
    y,
    n_estimators=35,
    learning_rate=0.12,
    max_depth=2,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=5.0,
)

pred = predict_tiny_xgb(model, X)
assert pred.shape == y.shape
assert squared_error_loss(y, pred) < squared_error_loss(y, y_hat0)

plt.figure(figsize=(8, 4))
plt.scatter(X[:, 0], y, s=18, alpha=0.55, label="observed")
plt.plot(X[:, 0], y_true, color="black", linewidth=2, label="noise-free pattern")
plt.plot(X[:, 0], pred, color="tab:red", linewidth=2.5, label="tiny XGB prediction")
plt.legend()
plt.title("Boosted trees learned as additive corrections")
plt.show()

print("Final loss:", squared_error_loss(y, pred))

## 9. Track Learning Over Rounds

A model that only reports the final prediction is hard to reason about. Real training workflows track metrics over boosting rounds.

Add a history list that records the loss after each tree. This will help you see underfitting, overfitting, and the effect of learning rate.

In [ ]:
# Edit original fit function and add history to the loss


model = fit_tiny_xgb_regressor(X, y, n_estimators=60, learning_rate=0.08, max_depth=2, min_child_weight=5.0)
history = model.loss_history
assert len(history) == 60
assert history[-1] < history[0]

plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("boosting round")
plt.ylabel("training loss")
plt.title("Training loss across boosting rounds")
plt.show()

## 10. Regularization Experiments

Now that you have a working learner, use it like a lab instrument.

Things to investigate:

- Larger `reg_lambda` shrinks leaf weights toward zero.
- Larger `gamma` requires a split to earn its complexity cost.
- Larger `min_child_weight` prevents tiny leaves.
- Larger `max_depth` makes individual trees more expressive.
- Smaller `learning_rate` usually needs more trees.

Read up on: bias-variance tradeoff, shrinkage, structural regularization, and early stopping.

In [ ]:
experiment_grid = [
    {"label": "low lambda", "reg_lambda": 0.0, "gamma": 0.0, "max_depth": 2},
    {"label": "high lambda", "reg_lambda": 10.0, "gamma": 0.0, "max_depth": 2},
    {"label": "high gamma", "reg_lambda": 1.0, "gamma": 2.0, "max_depth": 2},
    {"label": "deeper trees", "reg_lambda": 1.0, "gamma": 0.0, "max_depth": 4},
]

plt.figure(figsize=(9, 5))
plt.scatter(X[:, 0], y, s=14, alpha=0.35, color="gray", label="observed")

for cfg in experiment_grid:
    # TODO: fit a model using cfg values and plot its predictions.
    # Hint: pass cfg["reg_lambda"], cfg["gamma"], and cfg["max_depth"].
    preds = None
    plt.plot(X[:, 0], preds, linewidth=2, label=cfg["label"])

plt.legend()
plt.title("How regularization changes the learned function")
plt.show()

## 11. Stretch Goal: Logistic Loss

XGBoost is not only for regression. For binary classification, trees add to the raw log-odds score, also called the margin.

$$p = \sigma(margin) = \frac{1}{1 + e^{-margin}}$$

For binary log loss:

$$\ell(y, margin) = -y\log(p) - (1-y)\log(1-p)$$

The gradient and Hessian with respect to the margin are:

$$g = p - y, \quad h = p(1-p)$$

Notice the tree-building code does not care where `g` and `h` came from. That is the elegance of the abstraction.

In [ ]:
def sigmoid(z):
    # TODO: implement a numerically reasonable sigmoid.
    return None


def logloss_grad_hess(y_binary, margin):
    """Gradient and Hessian of binary log loss with respect to raw margin."""
    # TODO: compute p, then grad=p-y and hess=p*(1-p).
    p = None
    grad = None
    hess = None
    return grad, hess


assert np.isclose(sigmoid(0.0), 0.5)
gb, hb = logloss_grad_hess(np.array([0.0, 1.0]), np.array([0.0, 0.0]))
assert np.allclose(gb, np.array([0.5, -0.5]))
assert np.allclose(hb, np.array([0.25, 0.25]))
print("Logistic gradient and Hessian look right.")

## 12. Optional: Compare to Real XGBoost

If you have the `xgboost` package installed, compare this tiny implementation to the real library. Do not expect identical predictions; the production implementation has many extra details.

Look for conceptual matches instead:

- Both add trees sequentially.
- Both use gradient/Hessian statistics.
- Both expose `max_depth`, `learning_rate`, `reg_lambda`, `gamma`, and `min_child_weight`.
- Both become more flexible as you add trees or depth.

Read up on: exact vs approximate split finding, column subsampling, row subsampling, missing-value default directions, histogram algorithms, and sparsity-aware learning.

In [ ]:
try:
    from xgboost import XGBRegressor

    real_xgb = XGBRegressor(
        n_estimators=35,
        learning_rate=0.12,
        max_depth=2,
        reg_lambda=1.0,
        gamma=0.0,
        min_child_weight=5.0,
        objective="reg:squarederror",
        random_state=7,
    )
    real_xgb.fit(X, y)
    real_pred = real_xgb.predict(X)

    plt.figure(figsize=(8, 4))
    plt.scatter(X[:, 0], y, s=14, alpha=0.35, label="observed")
    plt.plot(X[:, 0], pred, linewidth=2, label="your tiny implementation")
    plt.plot(X[:, 0], real_pred, linewidth=2, label="real XGBoost")
    plt.legend()
    plt.title("Tiny implementation vs real XGBoost")
    plt.show()
except ImportError:
    print("xgboost is not installed. That is fine; this comparison is optional.")

## Final Reflection Prompts

Answer these in your own words after finishing the notebook:

1. Why does a leaf predict $-G/(H+\lambda)$ instead of the average residual?
2. What does the Hessian contribute beyond the gradient?
3. Why can `gamma` stop a split even when the child leaves improve training loss?
4. How does learning rate change the role of `n_estimators`?
5. Which part of this implementation would become too slow first on a large dataset?

When these feel clear, you understand the heart of XGBoost.